## Импорт библиотек

In [ ]:
import os
import gc
import pickle
import time

import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import VotingRegressor

from joblib import dump
from joblib import load

import lightgbm as lgb
import catboost as cb

import optuna
import matplotlib.pyplot as plt

from mlxtend.evaluate.time_series import GroupTimeSeriesSplit, plot_splits
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

In [ ]:
# Время старта работы ноутбука
notebook_starttime = time.time()

## Настройка: сабмит или локально

In [ ]:
# Ставим is_local в True, если локально работаем, если сабмитим - ставим в False
#is_local = False
is_local = False

# Ставим is_gpu в True, если будем работать на GPU, если на процессоре - ставим в False
is_gpu = True

# Ставим is_tuning в True, если запускаем подбор гиперпараметров в Optuna
is_tuning = False

# Ставим only_one_model = True, если обучается только дна модель без ансбамбля.
only_one_model = False
#only_one_model = True

# Раз во сколько циклов сабмита (дней) учить модели заново с учетом новый данных, добавленных к старым
# Скажем если учить заново каждые 30 циклов (после каждых 30 предсказанных дней) то ставим 30
learn_again_period = 2

# Начальная дата обучения модели
training_start_date = 'datetime >= "2022-01-01 00:00:00"'

In [ ]:
class MonthlyKFold:
    def __init__(self, n_splits=3):
        self.n_splits = n_splits
        
    def split(self, X, y, groups=None):
        dates = 12 * X["year"] + X["month"]
        timesteps = sorted(dates.unique().tolist())
        X = X.reset_index()
        
        for t in timesteps[-self.n_splits:]:
            idx_train = X[dates.values < t].index
            idx_test = X[dates.values == t].index
            
            yield idx_train, idx_test
            
    def get_n_splits(self, X, y, groups=None):
        return self.n_splits

In [ ]:
cv = MonthlyKFold()
CV = GroupTimeSeriesSplit(test_size=3, n_splits=3, window_type='rolling', shift_size=2)

## Feature Engineering sub

In [ ]:
def feature_eng(df_data, df_client, df_gas, df_electricity, df_forecast, df_historical, df_location, df_target, working_days):
    working_days = (
        working_days
        .with_columns(
            pl.col("date").cast(pl.Date)
        )
    )
    
    df_data = (
        df_data
        .with_columns(
            pl.col("datetime").cast(pl.Date).alias("date"),
        )
    )
    
    df_client = (
        df_client
        .with_columns(
            (pl.col("date") + pl.duration(days=2)).cast(pl.Date)
        )
    )
    
    df_gas = (
        df_gas
        .rename({"forecast_date": "date"})
        .with_columns(
            (pl.col("date") + pl.duration(days=1)).cast(pl.Date)
        )
    )
    
    df_electricity = (
        df_electricity
        .rename({"forecast_date": "datetime"})
        .with_columns(
            pl.col("datetime") + pl.duration(days=1)
        )
    )
    
    df_location = (
        df_location
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32)
        )
    )
    
    df_forecast = (
        df_forecast
        .rename({"forecast_datetime": "datetime"})
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32),
            #pl.col('datetime').dt.convert_time_zone("Europe/Bucharest").dt.replace_time_zone(None).cast(pl.Datetime("us")),
            pl.col('datetime').dt.replace_time_zone(None).cast(pl.Datetime("us"))
            #pl.col('datetime').cast(pl.Datetime)
        )
        .join(df_location, how="left", on=["longitude", "latitude"])
#         .drop("longitude", "latitude")
    )
    
    df_historical = (
        df_historical
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32),
            pl.col("datetime") + pl.duration(hours=37)
        )
        .join(df_location, how="left", on=["longitude", "latitude"])
#         .drop("longitude", "latitude")
    )
    
    df_forecast_date = (
        df_forecast
        .group_by("datetime").mean()
        .drop("county")
    )
    
    df_forecast_local = (
        df_forecast
        .filter(pl.col("county").is_not_null())
        .group_by("county", "datetime").mean()
    )
    
    df_historical_date = (
        df_historical
        .group_by("datetime").mean()
        .drop("county")
    )
    
    df_historical_local = (
        df_historical
        .filter(pl.col("county").is_not_null())
        .group_by("county", "datetime").mean()
    )
    # Объединение всех обработанных данных с основным датафреймом df_data
    df_data = (
        df_data
        .join(df_gas, on="date", how="left")
        .join(df_client, on=["county", "is_business", "product_type", "date"], how="left")
        .join(df_electricity, on="datetime", how="left")
        
        .join(df_forecast_date, on="datetime", how="left", suffix="_fd")
        .join(df_forecast_local, on=["county", "datetime"], how="left", suffix="_fl")
        .join(df_historical_date, on="datetime", how="left", suffix="_hd")
        .join(df_historical_local, on=["county", "datetime"], how="left", suffix="_hl")
        
        .join(df_forecast_date.with_columns(pl.col("datetime") + pl.duration(days=7)), on="datetime", how="left", suffix="_fdw")
        .join(df_forecast_local.with_columns(pl.col("datetime") + pl.duration(days=7)), on=["county", "datetime"], how="left", suffix="_flw")
        .join(df_historical_date.with_columns(pl.col("datetime") + pl.duration(days=7)), on="datetime", how="left", suffix="_hdw")
        .join(df_historical_local.with_columns(pl.col("datetime") + pl.duration(days=7)), on=["county", "datetime"], how="left", suffix="_hlw")
        
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=2)).rename({"target": "target_1"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=3)).rename({"target": "target_2"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=4)).rename({"target": "target_3"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=5)).rename({"target": "target_4"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=6)).rename({"target": "target_5"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=7)).rename({"target": "target_6"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=8)).rename({"target": "target_7"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=9)).rename({"target": "target_8"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=10)).rename({"target": "target_9"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=11)).rename({"target": "target_10"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=12)).rename({"target": "target_11"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=13)).rename({"target": "target_12"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=14)).rename({"target": "target_13"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        
        .join(working_days, on="date", how="left")
        # Создание категориальных признаков и тригонометрических функций времени
        .with_columns(
            pl.col("datetime").dt.ordinal_day().alias("dayofyear"), # Добавление номера дня в году
            pl.col("datetime").dt.hour().alias("hour"),# Добавление часа
            pl.col("datetime").dt.day().alias("day"),# Добавление дня
            pl.col("datetime").dt.weekday().alias("weekday"),# Добавление дня недели
            pl.col("datetime").dt.month().alias("month"),# Добавление месяца
            pl.col("datetime").dt.year().alias("year"),# Добавление года
        )
        # Приведение типов данных
        .with_columns(
            pl.concat_str("county", "is_business", "product_type", "is_consumption", separator="_").alias("category_1"),
        )
        
        .with_columns(
            (np.pi * pl.col("dayofyear") / 183).sin().alias("sin(dayofyear)"), # Тригонометрические функции для дня в году
            (np.pi * pl.col("dayofyear") / 183).cos().alias("cos(dayofyear)"),
            (np.pi * pl.col("hour") / 12).sin().alias("sin(hour)"),
            (np.pi * pl.col("hour") / 12).cos().alias("cos(hour)"),
        )
        
        .with_columns(
            pl.col(pl.Float64).cast(pl.Float32),
        )
         # Удаление ненужных колонок
        .drop("date", "hour", "dayofyear")
    )
    
    # return df_data, df_historical_local
    return df_data

In [ ]:
def to_pandas(X, y=None):
    cat_cols = ["county", "is_business", "product_type", "is_consumption", "category_1"]
    
    if y is not None:
        df = pd.concat([X.to_pandas(), y.to_pandas()], axis=1)
    else:
        df = X.to_pandas()    
    grouped_df = df.groupby(['category_1', 'year', 'month', 'day'])
    df = df.merge(grouped_df['target_1'].sum().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_sum'])
    df = df.merge(grouped_df['target_1'].mean().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_mean'])
    df = df.merge(grouped_df['target_6'].sum().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_sum'])
    df = df.merge(grouped_df['target_6'].mean().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_mean'])
    df = df.merge(grouped_df['target_13'].sum().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_sum'])
    df = df.merge(grouped_df['target_13'].mean().reset_index(), on=['category_1', 'year', 'month', 'day'], how='left', suffixes=['', '_mean'])
    
    df = df.set_index("row_id")
    df[cat_cols] = df[cat_cols].astype("category")
    
    df["target_mean"] = df[[f"target_{i}" for i in range(1, 13)]].mean(1)
    df["target_std"] = df[[f"target_{i}" for i in range(1, 13)]].std(1)
    df["target_ratio"] = df["target_6"] / (df["target_13"] + 1e-3)
    
    return df

In [ ]:
# для оптуны
# Диапазон гиперпараметров для модели обучающейся на данных is_consumption=1 
def lgb_objective_cons1(trial):
    params = {
        'device'            : trial.suggest_categorical('device', ['gpu']),
        'gpu_platform_id'   : trial.suggest_int('gpu_platform_id', 1, 1),
        'gpu_device_id'     : trial.suggest_int('gpu_device_id', 0, 0),
        'n_estimators'      : trial.suggest_int('n_estimators', 1000, 2000),
        'verbose'           : trial.suggest_int('verbose', -1, -1),
        'random_state'      : trial.suggest_int('random_state', 42, 42),
        'objective'         : trial.suggest_categorical('objective', ['l2']),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 50),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.1),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'colsample_bynode'  : trial.suggest_float('colsample_bynode', 0.1, 1.0),
        #'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-2, 20.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-2, 10.0),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-2, 20.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 4, 256),
        'max_depth'         : trial.suggest_int('max_depth', -1, -1),
        'max_bin'           : trial.suggest_int('max_bin', 32, 200),
    }
    
    model  = lgb.LGBMRegressor(**params)
    X      = df_train[df_train['is_consumption']==1].drop(columns=["target", "datetime"]).reset_index(drop=True)
    y      = df_train[df_train['is_consumption']==1]["target"].reset_index(drop=True)
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error')
    
    return -np.mean(scores)

# Диапазон гиперпараметров для модели обучающейся на данных is_consumption=0
def lgb_objective_cons0(trial):
    params = {
        'device'            : trial.suggest_categorical('device', ['gpu']),
        'gpu_platform_id'   : trial.suggest_int('gpu_platform_id', 1, 1),
        'gpu_device_id'     : trial.suggest_int('gpu_device_id', 0, 0),
        'n_estimators'      : trial.suggest_int('n_estimators', 1000, 2000),
        'verbose'           : trial.suggest_int('verbose', -1, -1),
        'random_state'      : trial.suggest_int('random_state', 42, 42),
        'objective'         : trial.suggest_categorical('objective', ['l2']),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 50),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.1),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'colsample_bynode'  : trial.suggest_float('colsample_bynode', 0.1, 1.0),
        #'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-2, 20.0),
        #'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-2, 20.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-2, 10.0),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 12, 20.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 4, 256),
        'max_depth'         : trial.suggest_int('max_depth', -1, -1),
        'max_bin'           : trial.suggest_int('max_bin', 32, 200),
    }
    
    model  = lgb.LGBMRegressor(**params)
    X      = df_train[df_train['is_consumption']==0].drop(columns=["target", "datetime"]).reset_index(drop=True)
    y      = df_train[df_train['is_consumption']==0]["target"].reset_index(drop=True)
    scores = cross_val_score(model, X, y, groups=groups, cv=cv, scoring='neg_mean_absolute_error')
    
    return -np.mean(scores)

## Global Variables

In [ ]:
root = "/kaggle/input/predict-energy-behavior-of-prosumers"

# Для локальных вычислений. Последний data_block_id тренировочной выборки
# А начиная со следующего data_block_id и до конца идет тест
train_end_data_block_id = 500

data_cols        = ['target', 'county', 'is_business', 'product_type', 'is_consumption', 'datetime', 'row_id']
# В df_data_cols колонки в таком порядке в каком они потом формируются в df_data
df_data_cols     = ['county', 'is_business', 'product_type', 'target', 'is_consumption', 'datetime', 'row_id']
client_cols      = ['product_type', 'county', 'eic_count', 'installed_capacity', 'is_business', 'date']
gas_cols         = ['forecast_date', 'lowest_price_per_mwh', 'highest_price_per_mwh']
electricity_cols = ['forecast_date', 'euros_per_mwh']
forecast_cols    = ['latitude', 'longitude', 'hours_ahead', 'temperature', 'dewpoint', 'cloudcover_high', 'cloudcover_low', 'cloudcover_mid', 'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'forecast_datetime', 'direct_solar_radiation', 'surface_solar_radiation_downwards', 'snowfall', 'total_precipitation']
historical_cols  = ['datetime', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure','cloudcover_total','cloudcover_low','cloudcover_mid','cloudcover_high','windspeed_10m','winddirection_10m','shortwave_radiation','direct_solar_radiation','diffuse_radiation','latitude','longitude']
location_cols    = ['longitude', 'latitude', 'county']
target_cols      = ['target', 'county', 'is_business', 'product_type', 'is_consumption', 'datetime']

save_path = None
load_path = None

## Исследование

In [ ]:
# Загрузка данных об энергопотреблении
train = pd.read_csv(os.path.join(root, "train.csv"))

# Создание сводной таблицы с средними значениями целевой переменной (target)
# для каждой комбинации даты, округа, типа продукта, бизнеса и потребления
pivot_train = train.pivot_table(
    index='datetime',
    columns=['county', 'product_type', 'is_business', 'is_consumption'],
    values='target',
    aggfunc='mean'
)

# Переименование колонок для удобства доступа и интерпретации
pivot_train.columns = ['county{}_productType{}_isBusiness{}_isConsumption{}'.format(*col) for col in pivot_train.columns.values]
pivot_train.index = pd.to_datetime(pivot_train.index)

pivot_train

### 2023 год 

In [ ]:
# Копирование сводной таблицы для визуализации
df_plot = pivot_train.copy()

# Нормализация данных для визуализации
df_plot = (df_plot - df_plot.min()) / (df_plot.max() - df_plot.min())

# Ресемплирование данных по дням и вычисление средних значений
df_plot_resampled_D = df_plot.resample('D').mean()

# Визуализация нормализованных данных с прозрачностью (alpha=0.1)
df_plot_resampled_D.loc['2022-7':].plot(alpha=0.1, color='green', figsize=(18, 6), legend=False)


In [ ]:
# Выбор колонок, соответствующих различным категориям потребления
columns_consumption_0 = df_plot_resampled_D.columns[df_plot_resampled_D.columns.str.contains('isConsumption0')]
columns_consumption_1 = df_plot_resampled_D.columns[df_plot_resampled_D.columns.str.contains('isConsumption1')]

# Создание фигуры для визуализации
plt.figure(figsize=(15, 6))

# Создание пустых линий для легенды
plt.plot([], color='red', label='is_Consumption = 1')  # Изменено на желтый цвет
plt.plot([], color='black', label='is_Consumption = 0')   # Изменено на черный цвет

# Отображение легенды
plt.legend()

# Визуализация данных для 'is_Consumption = 0' черным цветом
for column in columns_consumption_0:
    df_plot_resampled_D.loc['2022-7':, column].plot(alpha=0.1, color='black', legend=False)  # Изменено на черный

# Визуализация данных для 'is_Consumption = 1' желтым цветом
for column in columns_consumption_1:
    df_plot_resampled_D.loc['2022-7':, column].plot(alpha=0.1, color='red', legend=False)  # Изменено на желтый

# Отображение графика
plt.show()




## Подготовка данных

### Запись тестовых и тренировочных csv файлов

In [ ]:
if is_local:
    # Если выполняем локально
    train_path = 'train'
    if not os.path.exists(train_path):
        # Создание каталога, если его нет
        os.makedirs(train_path)
    test_path = 'example_test_files'
    if not os.path.exists(test_path):
        # Создание каталога, если его нет
        os.makedirs(test_path)
else:
    # Если сабмит
    train_path = root
# Путь, куда запишем csv файлы для теста

In [ ]:
# Разделяет датафрейм на тренировочную и тестовую часть
# Возвращает часть датафрейма для тренировки, тестовую часть датафрейма записывает в каталог с тестами
def split_train_test(filename):
    df = pd.read_csv(os.path.join(root, filename))
    
    #Запишем часть данных для теста
    test_df = df[df["data_block_id"] > train_end_data_block_id]
    if (filename =="train.csv"):
        # Берем только те ячейки где target был не нулевым
        test_df = test_df[test_df["target"].notnull()]
        
    test_df.to_csv(os.path.join(test_path, filename), index=False)

    #Запишем часть данных для трейна
    train_df = df[df["data_block_id"] <= train_end_data_block_id]
    train_df.to_csv(os.path.join(train_path, filename), index=False)

# Доводим до ума тестовые таблицы чтобы они были точно такие как в реальном сабмите
def test_dfs_tune():
    # Делаем таблицу revealed_targets.csv
    df = pd.read_csv(os.path.join(root, "train.csv"))
    df = df[df["data_block_id"] > train_end_data_block_id - 2]
    df["data_block_id"] += 2
    df = df[df["target"].notnull()]
    df.to_csv(os.path.join(test_path, 'revealed_targets.csv'), index=False)
    
    # Делаем таблицу test.csv
    df = pd.read_csv(os.path.join(test_path, "train.csv"))
    df.rename(columns={'datetime': 'prediction_datetime'}, inplace=True)
    df.drop('target', axis=1, inplace=True)
    df['currently_scored'] = False
    df.to_csv(os.path.join(test_path, 'test.csv'), index=False)
    
    # Делаем таблицу sample_submission.csv
    selected_columns = ['row_id', 'data_block_id']
    df = df[selected_columns]
    df['target'] = 0
    df.to_csv(os.path.join(test_path, 'sample_submission.csv'), index=False)

# Сборка разделения файлов
def make_split():
    # csv файлы которые будем делить:
    csv_names = ["train.csv", "client.csv", "gas_prices.csv", "electricity_prices.csv", "forecast_weather.csv", "historical_weather.csv"]
    for csv_name in csv_names:
        split_train_test(csv_name)
    # Доделываем тестовые таблицы
    test_dfs_tune()

In [ ]:
%%time

# Создаем файлы csv c тренировочными и тестовыми таблицами
if is_local:
    # Пока отключил создание тестовых файлом. У меня локально они есть
    # make_split()
    pass

### Data I/O

In [ ]:
%%time
df_data        = pl.read_csv(os.path.join(train_path, "train.csv"), columns=data_cols, try_parse_dates=True)
df_client      = pl.read_csv(os.path.join(train_path, "client.csv"), columns=client_cols, try_parse_dates=True)
df_gas         = pl.read_csv(os.path.join(train_path, "gas_prices.csv"), columns=gas_cols, try_parse_dates=True)
df_electricity = pl.read_csv(os.path.join(train_path, "electricity_prices.csv"), columns=electricity_cols, try_parse_dates=True)
df_forecast    = pl.read_csv(os.path.join(train_path, "forecast_weather.csv"), columns=forecast_cols, try_parse_dates=True)
df_historical  = pl.read_csv(os.path.join(train_path, "historical_weather.csv"), columns=historical_cols, try_parse_dates=True)
#df_location    = pl.read_csv(os.path.join(root, "weather_station_to_county_mapping.csv"), columns=location_cols, try_parse_dates=True)
df_location    = pl.read_csv('/kaggle/input/locations/county_lon_lats.csv', columns=location_cols, try_parse_dates=True)
df_target      = df_data.select(target_cols)
working_days   = pl.read_csv('/kaggle/input/working-days/working_days.csv', try_parse_dates=True)

schema_data        = df_data.schema
schema_client      = df_client.schema
schema_gas         = df_gas.schema
schema_electricity = df_electricity.schema
schema_forecast    = df_forecast.schema
schema_historical  = df_historical.schema
schema_target      = df_target.schema

### Feature Engineering

In [ ]:
X, y = df_data.drop("target"), df_data.select("target")

X = feature_eng(X, df_client, df_gas, df_electricity, df_forecast, df_historical, df_location, df_target, working_days)

df_train = to_pandas(X, y)

In [ ]:
df_train.head()

In [ ]:
# df_train.to_csv('test.csv', index=False)

In [ ]:
# рорпрпфи98789

In [ ]:
#print(df_train.columns.to_list())
#pd.set_option('display.max_rows', None)
#df_train.loc[174]

In [ ]:
df_train = df_train[df_train["target"].notnull()].query(training_start_date)

In [ ]:
# # Извлечение столбцов 'county' и 'target'
# new_df = df_train[['county', 'target']]

# # Вывод первых нескольких строк нового DataFrame для проверки
# new_df.head()

In [ ]:
# # Извлечение столбцов 'county', 'target', 'latitude' и 'longitude'
new_df = df_train[['county', 'target', 'latitude', 'longitude','datetime','is_consumption']]
# # Извлечение столбцов 'county', 'target', 'latitude' и 'longitude'
# new_df = df_train[['county', 'target', 'latitude', 'longitude']]

# Вывод первых нескольких строк нового DataFrame для проверки
new_df.head()


In [ ]:
# new_df.to_csv('new_df1.csv', index=False)

In [ ]:
len(new_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Преобразуем datetime в формат datetime, если это еще не сделано
df_train['datetime'] = pd.to_datetime(df_train['datetime'])

# Отсортируем DataFrame по времени
df_train.sort_values(by='datetime', inplace=True)

# Получаем уникальные значения county и сортируем их в порядке возрастания
counties = df_train['county'].cat.categories
sorted_counties = sorted(counties)

# Находим количество уникальных значений county
num_counties = len(sorted_counties)

# Создаем фигуру с подграфиками
fig, axs = plt.subplots(num_counties, 2, figsize=(15, 5 * num_counties), gridspec_kw={'width_ratios': [4, 1]})

# В цикле создаем график для каждого county
for i, county in enumerate(sorted_counties):
    # Выбираем данные для данного county
    county_data = df_train[df_train['county'] == county]
    
    # Настраиваем линейный график
    axs[i, 0].plot(county_data['datetime'], county_data['target'], label=f'Target')
    axs[i, 0].set_title(f'Изменение таргета для {county}')
    axs[i, 0].set_xlabel('Дата')
    axs[i, 0].set_ylabel('Таргет')
    axs[i, 0].legend()
    
    # Настраиваем boxplot
    axs[i, 1].boxplot(county_data['target'], vert=True)
    axs[i, 1].set_title(f'Распределение таргета для {county}')
    axs[i, 1].set_xlabel('Таргет')
    axs[i, 1].set_yticks([])  # Удаляем метки с оси y

# Автоматическое уплотнение расположения подграфиков
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Преобразуем datetime в формат datetime, если это еще не сделано
df_train['datetime'] = pd.to_datetime(df_train['datetime'])

# Отсортируем DataFrame по времени
df_train.sort_values(by='datetime', inplace=True)

# Получаем уникальные значения county и сортируем их
sorted_counties = sorted(df_train['county'].unique())

# Находим количество уникальных значений county
num_counties = len(sorted_counties)

# Создаем фигуру с подграфиками
fig, axs = plt.subplots(num_counties, 2, figsize=(15, 5 * num_counties), gridspec_kw={'width_ratios': [4, 1]})

# В цикле создаем график для каждого county
for i, county in enumerate(sorted_counties):
    # Выбираем данные для данного county
    county_data = df_train[df_train['county'] == county]
    
    # Разделяем данные на основе 'is_consumption'
    data_cons_0 = county_data[county_data['is_consumption'] == 0]
    data_cons_1 = county_data[county_data['is_consumption'] == 1]

    # Построение линейных графиков с прозрачностью 0.1
    axs[i, 0].plot(data_cons_0['datetime'], data_cons_0['target'], 'g', alpha=0.4, label='Not Consumption (0)')
    axs[i, 0].plot(data_cons_1['datetime'], data_cons_1['target'], 'r', alpha=0.4, label='Consumption (1)')
    axs[i, 0].set_title(f'Изменение таргета для {county}')
    axs[i, 0].set_xlabel('Дата')
    axs[i, 0].set_ylabel('Таргет')
    axs[i, 0].legend()
    
    # Настраиваем boxplot
    axs[i, 1].boxplot([data_cons_0['target'], data_cons_1['target']], positions=[1, 2], vert=True)
    axs[i, 1].set_title(f'Распределение таргета для {county}')
    axs[i, 1].set_xlabel('Таргет')
    axs[i, 1].set_xticklabels(['0', '1']) # Обозначаем метки для двух групп
    axs[i, 1].set_yticks([])  # Удаляем метки с оси y

# Автоматическое уплотнение расположения подграфиков
plt.tight_layout()
plt.show()